In [1]:
import pandas as pd
import utils
from Bio import SeqIO
import pickle
import argparse
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import T5EncoderModel, T5Tokenizer
from config import config
from dataset import MyDataset
from torch import optim, nn
from torch.utils.data import TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.ensemble import RandomForestClassifier
from model import Cnn
from loss import *
from torch.utils.data import DataLoader
from torch.optim import lr_scheduler
from utils import *
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, matthews_corrcoef
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
utils.seed_everything(config.seed)

# Binary (PVP Indentification)

In [3]:
label_dict = {'non-PVP': 0, 'PVP': 1}
test_file_name = 'data/split_smi/binary/test_'
threshold = ['40', '50', '60', '70', '80', '90']

In [4]:
# get_embedding
embedding_df = pd.read_excel(config.embedding_file)
embedding_df.set_index("id", inplace=True)

In [5]:
Test_acc = []
Precision = []
Recall = []
F1 = []
Mcc = []
Sensitivity = []
Specificity = []



for item in threshold:
    test_file = test_file_name + item + '.csv'
    model_path = 'model/split_smi/binary/binary_model_' + item + '.pth'
    
    test_df = pd.read_csv(test_file)
    test_proteins, test_ids, test_labels = [], [], []
    
    for index, row in test_df.iterrows():
        test_proteins.append(row['sequence'])
        test_ids.append(row['accession'].split('.')[0])
        test_labels.append(label_dict[row['label']])
        

    y_test = np.array(test_labels)
    test_data = MyDataset(test_ids, test_labels)
    test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)
    
    
    model = Cnn().to(config.device)
    model.load_state_dict(torch.load(model_path))
    
    
    # test
    labels, test_epoch_preds = test(model, test_dataloader, embedding_df)

    # results
    test_acc = accuracy_score(y_test, test_epoch_preds)
    precision = precision_score(y_test, test_epoch_preds)
    recall = recall_score(y_test, test_epoch_preds)
    f1 = f1_score(y_test, test_epoch_preds)
    mcc = matthews_corrcoef(y_test, test_epoch_preds)
    tn, fp, fn, tp = confusion_matrix(y_test, test_epoch_preds).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    
    Test_acc.append(test_acc)
    Precision.append(precision)
    Recall.append(recall)
    F1.append(f1)
    Mcc.append(mcc)
    Sensitivity.append(sensitivity)
    Specificity.append(specificity)
new_df = pd.DataFrame({"threshold": threshold, "ACC": Test_acc, "Precision": Precision, "Recall": Recall, "F1": F1, "MCC": Mcc, "Sensitivity": Sensitivity, "Specificity": Specificity})
new_df.to_csv(f"results/split_smi/binary_predict_results.csv", index=False)

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: La

# Multi-class (PVP function annotation)

In [6]:
label_dict = {'minor_capsid':0, 'tail_fiber':1, 'major_tail':2, 'portal':3, 'minor_tail':4, 'baseplate':5, 'major_capsid':6}
test_file_name = 'data/split_smi/multi_class/test_'

In [7]:
Test_acc = []
Precision = []
Recall = []
F1 = []
Mcc = []
Sensitivity = []
Specificity = []

for item in threshold:
    test_file = test_file_name + item + '.csv'
    model_path = 'model/split_smi/multi_class/muti_model_' + item + '.pth'
    
    test_df = pd.read_csv(test_file)
    test_proteins, test_ids, test_labels = [], [], []
    
    for index, row in test_df.iterrows():
        test_proteins.append(row['sequence'])
        test_ids.append(row['accession'].split('.')[0])
        test_labels.append(label_dict[row['label']])
        
    y_test = np.array(test_labels)
    test_data = MyDataset(test_ids, test_labels)
    test_dataloader = DataLoader(test_data, shuffle=False, batch_size=config.batch_size)
    
    model = Cnn_muti().to(config.device)
    model.load_state_dict(torch.load(model_path))
    
    # test
    labels, test_epoch_preds = test_muti(model, test_dataloader, embedding_df)

    # results
    test_acc = accuracy_score(y_test, test_epoch_preds)
    precision = precision_score(y_test, test_epoch_preds, average='weighted')
    recall = recall_score(y_test, test_epoch_preds, average='weighted')
    f1 = f1_score(y_test, test_epoch_preds, average='weighted')
    mcc = matthews_corrcoef(y_test, test_epoch_preds)
    
    Test_acc.append(test_acc)
    Precision.append(precision)
    Recall.append(recall)
    F1.append(f1)
    Mcc.append(mcc)
new_df = pd.DataFrame({"threshold": threshold, "ACC": Test_acc, "Precision": Precision, "Recall": Recall, "F1": F1, "MCC": Mcc})
new_df.to_csv(f"results/split_smi/multi_predict_results.csv", index=False)

/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
/home/oyh/anaconda3/envs/pytorch/lib/python3.8/site-packages/torch/nn/modules/lazy.py:178: UserWarning: La